# MovieLens-1M Recommendation System: Popularity vs. SVD/ALS vs. Neural Collaborative Filtering

This notebook provides an interactive, reproducible walkthrough of:
1. **Exploratory Data Analysis (EDA)** and per-user chronological `80/10/10` splitting on MovieLens-1M.
2. **Model Inspection**: Popularity (IMDB Weighted Rating), Matrix Factorization (`Biased SVD` & `Implicit ALS`), and Neural Collaborative Filtering (`GMF`, `MLP`, `NeuMF (Scratch)`, `NeuMF (Pre-trained)`).
3. **Full-Catalog Evaluation** (`RMSE`, `Precision@10`, `Recall@10`, `NDCG@10`) and **Error / Bias Analysis** (Cold-start, Gray-sheep, Popularity Bias, Catalog Coverage).

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import eda_summary, load_movielens, to_csr
from src.evaluate import load_all_models

## 1. Dataset Statistics & Temporal Split Verification

In [ ]:
data = load_movielens(ROOT / 'data' / 'ml-1m')
summary = eda_summary(data)
pd.Series({k: v for k, v in summary.items() if not isinstance(v, dict)})

## 2. Benchmark Results on Held-Out Test Split

In [ ]:
metrics_path = ROOT / 'results' / 'metrics.json'
if metrics_path.exists():
    df_metrics = pd.DataFrame(json.loads(metrics_path.read_text(encoding='utf-8')))
    display(df_metrics)
else:
    print('Run `python -m src.evaluate` first to generate results/metrics.json.')

## 3. User-Cohort Error Analysis & Popularity Bias

In [ ]:
err_path = ROOT / 'results' / 'error_analysis.json'
if err_path.exists():
    err = json.loads(err_path.read_text(encoding='utf-8'))
    display(pd.DataFrame(err['cohorts']))
    display(pd.DataFrame(err['popularity_bias']))

## 4. Qualitative Top-10 Recommendation Inspection

In [ ]:
models = load_all_models(data, ROOT / 'saved_models', device='cpu', demo_subset=True)
seen = to_csr(pd.concat([data.train, data.val], ignore_index=True), data.n_users, data.n_items)
uid = 0
for name, mdl in models.items():
    s = mdl.score_all(torch.tensor([uid]))[0].clone()
    s[seen[uid].indices] = -float('inf')
    top10 = s.topk(10).indices.numpy()
    print(f'=== {name} Top-10 for User #{uid} ===')
    display(data.movies.iloc[top10][['title', 'genres']])